In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Add comments to all schemas in the catalog
spark.sql(f"COMMENT ON SCHEMA {catalog}.landing IS 'Raw data landing zone. Source files (CSV, JSON) arrive in the landing Unity Catalog volume. All unprocessed, raw data lives here before ingestion into bronze.'")
spark.sql(f"COMMENT ON SCHEMA {catalog}.bronze IS 'Raw Delta tables ingested from landing volume files. No filtering or transformation applied — data is stored as-is in Delta format.'")
spark.sql(f"COMMENT ON SCHEMA {catalog}.silver IS 'Cleaned and validated tables from bronze. Also contains quarantine tables for records that fail validation rules and data quality checks.'")
spark.sql(f"COMMENT ON SCHEMA {catalog}.gold IS 'Business-level aggregated data. All dimension tables (dim_*), fact tables (fact_*), and business KPI views live here for analytics and reporting.'")
spark.sql(f"COMMENT ON SCHEMA {catalog}.ops IS 'Operational support schema. Holds Auto Loader checkpoint locations, operational metadata, data quality check tables, cross-verification tables, and other pipeline operational data.'")

display(spark.sql(f"SELECT schema_name, comment FROM {catalog}.information_schema.schemata WHERE schema_name IN ('landing','bronze','silver','gold','ops') ORDER BY schema_name"))

In [0]:
# Comment on all silver tables and their columns based on the VoltPath business scenario

# ── silver.stations ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.stations IS 'Cleaned station master data. Station names title-cased, city names standardised, opened_on converted to DATE. Ten stations across Jaipur, Delhi, Mumbai, Bengaluru and Pune.'")
spark.sql(f"ALTER TABLE {catalog}.silver.stations ALTER COLUMN station_id COMMENT 'Unique identifier for each charging station'")
spark.sql(f"ALTER TABLE {catalog}.silver.stations ALTER COLUMN station_name COMMENT 'Human-readable name of the station (title-cased)'")
spark.sql(f"ALTER TABLE {catalog}.silver.stations ALTER COLUMN city COMMENT 'City where the station is located (Jaipur, Delhi, Mumbai, Bengaluru, Pune)'")
spark.sql(f"ALTER TABLE {catalog}.silver.stations ALTER COLUMN site_type COMMENT 'Type of site location (e.g. HIGHWAY, MALL, PETROL PUMP)'")
spark.sql(f"ALTER TABLE {catalog}.silver.stations ALTER COLUMN opened_on COMMENT 'Date the station was opened for public use'")

# ── silver.chargers ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.chargers IS 'Cleaned charger data maintained via SCD Type 1 MERGE on charger_id. Connector types uppercased, max_kw as DECIMAL(6,1), is_active as BOOLEAN. Thirty chargers (three per station).'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN charger_id COMMENT 'Unique identifier for each charger'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN station_id COMMENT 'Foreign key to silver.stations identifying which station this charger belongs to'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN connector_type COMMENT 'Type of connector (CCS2 or TYPE2), uppercased for consistent joins'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN max_kw COMMENT 'Maximum power output of the charger in kW (DECIMAL(6,1))'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN is_active COMMENT 'Whether the charger is currently operational (true/false)'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN commissioned_on COMMENT 'Date the charger was commissioned and ready for use'")
spark.sql(f"ALTER TABLE {catalog}.silver.chargers ALTER COLUMN last_changed_snapshot_date COMMENT 'Date of the weekly snapshot in which this charger record last changed'")

# ── silver.tariffs ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.tariffs IS 'Cleaned tariff data. Connector types and time bands uppercased. Four rows (CCS2/TYPE2 x PEAK/OFF_PEAK). Used for billing calculations in gold layer.'")
spark.sql(f"ALTER TABLE {catalog}.silver.tariffs ALTER COLUMN connector_type COMMENT 'Type of connector this tariff applies to (CCS2 or TYPE2), uppercased'")
spark.sql(f"ALTER TABLE {catalog}.silver.tariffs ALTER COLUMN time_band COMMENT 'Time period for this tariff rate (PEAK or OFF_PEAK), uppercased'")
spark.sql(f"ALTER TABLE {catalog}.silver.tariffs ALTER COLUMN rate_per_kwh COMMENT 'Energy charge rate per kWh consumed (DECIMAL(10,2))'")
spark.sql(f"ALTER TABLE {catalog}.silver.tariffs ALTER COLUMN idle_fee_per_min COMMENT 'Fee charged per minute of idle time after the free period (DECIMAL(10,2))'")
spark.sql(f"ALTER TABLE {catalog}.silver.tariffs ALTER COLUMN free_idle_minutes COMMENT 'Number of free idle minutes before the idle fee applies (INT)'")

# ── silver.members ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.members IS 'Member history maintained as SCD Type 2. Plan and city changes create a new version row; name, email and phone changes overwrite in place. Two-hundred members across BASIC, PRO and FLEET plans.'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN member_id COMMENT 'Unique identifier for each registered member'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN full_name COMMENT 'Full name of the member'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN email COMMENT 'Email address of the member (lowercased)'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN phone COMMENT 'Phone number of the member (last 10 digits)'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN city COMMENT 'City where the member is located (title-cased)'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN plan COMMENT 'Membership plan (BASIC, PRO or FLEET), uppercased — determines discount on charging fees'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN joined_on COMMENT 'Date the member joined VoltPath'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN effective_from COMMENT 'Date from which this version of the member record is valid (SCD Type 2)'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN effective_to COMMENT 'Date until which this version is valid; null for the current version (SCD Type 2)'")
spark.sql(f"ALTER TABLE {catalog}.silver.members ALTER COLUMN is_current COMMENT 'Whether this is the current version of the member record (true/false)'")

# ── silver.sessions ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.sessions IS 'Cleaned and validated charging sessions. Deduplicated by keeping the latest version per session_id (using updated_at). Energy, duration and amount fields typed properly. Time band derived from started_at. Five-hundred-twenty-four sessions.'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN session_id COMMENT 'Unique identifier for each charging session'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN charger_id COMMENT 'Foreign key to silver.chargers identifying which charger was used'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN member_id COMMENT 'Foreign key to silver.members; null for guest (non-member) sessions'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN started_at COMMENT 'Start timestamp of the charging session'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN ended_at COMMENT 'End timestamp of the charging session'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN session_date COMMENT 'Date portion of started_at, used for aggregation and date-key joins'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN status COMMENT 'Status of the session (COMPLETED, FAILED or STARTED)'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN time_band COMMENT 'Time band derived from started_at (PEAK or OFF_PEAK) for tariff lookup'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN energy_kwh COMMENT 'Energy consumed during the session in kWh (DECIMAL(10,3)), calculated from meter end_kwh minus start_kwh'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN duration_minutes COMMENT 'Duration of the charging session in minutes (INT), derived from ended_at minus started_at'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN idle_minutes COMMENT 'Minutes the vehicle remained idle after charging completed (INT)'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN amount_paid COMMENT 'Amount paid by the driver in INR (DECIMAL(10,2)), from the session payment struct'")
spark.sql(f"ALTER TABLE {catalog}.silver.sessions ALTER COLUMN updated_at COMMENT 'Timestamp of the last update to the session record; used to pick the latest version during deduplication'")

# ── silver.session_faults ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.session_faults IS 'Exploded fault records from the faults array in bronze.sessions. One row per fault per session. Used for charger health and reliability analysis.'")
spark.sql(f"ALTER TABLE {catalog}.silver.session_faults ALTER COLUMN session_id COMMENT 'Foreign key to silver.sessions identifying which session had the fault'")
spark.sql(f"ALTER TABLE {catalog}.silver.session_faults ALTER COLUMN fault_seq COMMENT 'Sequence number of the fault within the session (1-based)'")
spark.sql(f"ALTER TABLE {catalog}.silver.session_faults ALTER COLUMN fault_code COMMENT 'Code identifying the type of fault that occurred'")

# ── silver.quarantine_sessions ──
spark.sql(f"COMMENT ON TABLE {catalog}.silver.quarantine_sessions IS 'Sessions that failed validation rules (e.g. missing charger_id, invalid timestamps, negative energy). Contains the reject reason and the raw JSON record for manual review and reprocessing.'")
spark.sql(f"ALTER TABLE {catalog}.silver.quarantine_sessions ALTER COLUMN session_id COMMENT 'Identifier of the session that was rejected'")
spark.sql(f"ALTER TABLE {catalog}.silver.quarantine_sessions ALTER COLUMN reject_reason COMMENT 'Human-readable reason why the session failed validation'")
spark.sql(f"ALTER TABLE {catalog}.silver.quarantine_sessions ALTER COLUMN updated_at COMMENT 'Updated_at value from the rejected session record'")
spark.sql(f"ALTER TABLE {catalog}.silver.quarantine_sessions ALTER COLUMN raw_record COMMENT 'Full raw JSON of the rejected session, kept for manual review and potential reprocessing'")
spark.sql(f"ALTER TABLE {catalog}.silver.quarantine_sessions ALTER COLUMN quarantined_at COMMENT 'Timestamp when the session was moved to quarantine'")

print("All silver table and column comments added successfully.")

In [0]:
# Comment on all gold tables, the KPI view and their columns based on the VoltPath business scenario

# ── gold.dim_date ──
spark.sql(f"COMMENT ON TABLE {catalog}.gold.dim_date IS 'Date dimension covering 1-10 April 2026. date_key is INT (yyyyMMdd) for efficient fact-table joins. Thirty rows, one per day.'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_date ALTER COLUMN date_key COMMENT 'Integer date key in yyyyMMdd format (e.g. 20260401); primary key for joining to fact tables'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_date ALTER COLUMN calendar_date COMMENT 'Actual calendar date'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_date ALTER COLUMN day_name COMMENT 'Day of the week (e.g. Monday)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_date ALTER COLUMN iso_week COMMENT 'ISO week number of the year'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_date ALTER COLUMN is_weekend COMMENT 'Whether the date falls on a weekend (true/false)'")

# ── gold.dim_station ──
spark.sql(f"COMMENT ON TABLE {catalog}.gold.dim_station IS 'Station dimension. Cleaned station attributes for analytics. Ten rows.'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_station ALTER COLUMN station_id COMMENT 'Unique identifier for each charging station'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_station ALTER COLUMN station_name COMMENT 'Human-readable name of the station (title-cased)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_station ALTER COLUMN city COMMENT 'City where the station is located (Jaipur, Delhi, Mumbai, Bengaluru, Pune)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_station ALTER COLUMN site_type COMMENT 'Type of site location (e.g. HIGHWAY, MALL, PETROL PUMP)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_station ALTER COLUMN opened_on COMMENT 'Date the station was opened for public use'")

# ── gold.dim_charger ──
spark.sql(f"COMMENT ON TABLE {catalog}.gold.dim_charger IS 'Charger dimension with station city denormalised from dim_station. Thirty rows (three chargers per station).'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN charger_id COMMENT 'Unique identifier for each charger'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN station_id COMMENT 'Foreign key to gold.dim_station identifying which station this charger belongs to'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN connector_type COMMENT 'Type of connector (CCS2 or TYPE2)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN max_kw COMMENT 'Maximum power output of the charger in kW (DECIMAL(6,1))'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN is_active COMMENT 'Whether the charger is currently operational (true/false)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN commissioned_on COMMENT 'Date the charger was commissioned and ready for use'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_charger ALTER COLUMN station_city COMMENT 'City of the parent station, denormalised from dim_station for convenient filtering'")

# ── gold.dim_member ──
spark.sql(f"COMMENT ON TABLE {catalog}.gold.dim_member IS 'Member dimension with surrogate key (member_sk). Only the current version of each member from silver.members (is_current = true). Two-hundred rows.'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN member_sk COMMENT 'Surrogate key for the member dimension (BIGINT NOT NULL); used for fact-table joins'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN member_id COMMENT 'Natural business identifier for the member'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN full_name COMMENT 'Full name of the member'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN email COMMENT 'Email address of the member (lowercased)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN phone COMMENT 'Phone number of the member (last 10 digits)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN city COMMENT 'City where the member is located (title-cased)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN plan COMMENT 'Membership plan (BASIC, PRO or FLEET) — determines discount on charging fees'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN joined_on COMMENT 'Date the member joined VoltPath'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN effective_from COMMENT 'Date from which this version of the member record is valid (SCD Type 2)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN effective_to COMMENT 'Date until which this version is valid; null for the current version (SCD Type 2)'")
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN is_current COMMENT 'Whether this is the current version of the member record (true/false)'")

# ── gold.fact_sessions ──
spark.sql(f"COMMENT ON TABLE {catalog}.gold.fact_sessions IS 'Session fact table with full billing breakdown. Joins dim_date on date_key, dim_charger on charger_id, dim_member on member_sk. Only COMPLETED sessions receive billing values; FAILED and STARTED sessions get 0.00 or NULL. Five-hundred-twenty-four rows.'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN session_id COMMENT 'Unique identifier for each charging session'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN date_key COMMENT 'Foreign key to gold.dim_date (INT yyyyMMdd); derived from session_date'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN charger_id COMMENT 'Foreign key to gold.dim_charger identifying which charger was used'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN station_id COMMENT 'Foreign key to gold.dim_station, denormalised from dim_charger'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN member_id COMMENT 'Natural key for the member; null for guest (non-member) sessions'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN status COMMENT 'Status of the session (COMPLETED, FAILED or STARTED)'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN time_band COMMENT 'Time band during which the session started (PEAK or OFF_PEAK); used for tariff lookup'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN energy_kwh COMMENT 'Energy consumed during the session in kWh (DECIMAL(12,3))'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN duration_minutes COMMENT 'Duration of the charging session in minutes'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN idle_minutes COMMENT 'Minutes the vehicle remained idle after charging completed'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN amount_paid COMMENT 'Amount paid by the driver in INR (DECIMAL(12,2)); from the session payment struct'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN member_sk COMMENT 'Surrogate key to gold.dim_member; null for guest sessions'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN plan_at_session COMMENT 'Membership plan effective at the time of the session (BASIC, PRO, FLEET or GUEST)'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN rate_per_kwh COMMENT 'Tariff rate per kWh applied to this session based on connector type and time band'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN energy_charge COMMENT 'Raw energy charge before discount (energy_kwh x rate_per_kwh) (DECIMAL(12,2))'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN plan_discount COMMENT 'Discount amount based on membership plan (BASIC 0%, PRO 10%, FLEET 15%, GUEST 0%) (DECIMAL(12,2))'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN idle_fee COMMENT 'Idle fee charged for minutes beyond the free idle period (DECIMAL(12,2))'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN billed_amount COMMENT 'Total amount VoltPath billed to the driver: energy_charge - plan_discount + idle_fee (DECIMAL(12,2))'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN billing_status COMMENT 'Whether the billed amount matches what the driver paid (OK or MISMATCH)'")
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions ALTER COLUMN fault_count COMMENT 'Number of faults recorded during this session'")

# ── gold.agg_daily_station ──
spark.sql(f"COMMENT ON TABLE {catalog}.gold.agg_daily_station IS 'Daily per-station aggregation table. Fifty rows (five dates x ten stations). Used for operational dashboards to see which chargers are busy or failing.'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN session_date COMMENT 'Date of the aggregated sessions'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN station_id COMMENT 'Foreign key to gold.dim_station'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN city COMMENT 'City of the station, denormalised for filtering'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN completed_sessions COMMENT 'Count of sessions with status COMPLETED on this date at this station'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN failed_sessions COMMENT 'Count of sessions with status FAILED on this date at this station'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN energy_kwh COMMENT 'Total energy delivered in kWh across all sessions (DECIMAL(12,3))'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN billed_amount COMMENT 'Total amount billed by VoltPath in INR (DECIMAL(12,2))'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN amount_paid COMMENT 'Total amount paid by drivers in INR (DECIMAL(12,2))'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN mismatch_sessions COMMENT 'Count of sessions where billed_amount did not match amount_paid'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN fault_count COMMENT 'Total faults recorded across all sessions at this station on this date'")
spark.sql(f"ALTER TABLE {catalog}.gold.agg_daily_station ALTER COLUMN utilisation_pct COMMENT 'Percentage of active hours used for charging at this station (DECIMAL(5,2))'")

# ── gold.v_kpi_summary ──
spark.sql(f"COMMENT ON VIEW {catalog}.gold.v_kpi_summary IS 'Single-row view with headline business KPIs for the VoltPath executive dashboard. Shows overall success rate, energy delivered, billed vs paid revenue, leakage, and active members.'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.completed_sessions IS 'Total sessions that completed successfully'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.failed_sessions IS 'Total sessions that failed'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.success_rate_pct IS 'Percentage of sessions that completed successfully (DECIMAL(5,2))'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.total_energy_kwh IS 'Total energy delivered across all completed sessions in kWh (DECIMAL(12,3))'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.total_billed IS 'Total amount billed by VoltPath in INR (DECIMAL(12,2))'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.total_paid IS 'Total amount paid by drivers in INR (DECIMAL(12,2))'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.revenue_leakage IS 'Difference between total_billed and total_paid in INR (DECIMAL(12,2)); positive means drivers underpaid'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.mismatch_sessions IS 'Count of sessions where billed_amount did not match amount_paid'")
spark.sql(f"COMMENT ON COLUMN {catalog}.gold.v_kpi_summary.active_members IS 'Count of distinct members who had at least one charging session'")

print("All gold table, view and column comments added successfully.")

In [0]:
# Comment on all bronze tables and their columns based on the VoltPath business scenario

# ── bronze.stations ──
spark.sql(f"COMMENT ON TABLE {catalog}.bronze.stations IS 'Raw station master data ingested from a single CSV file. VoltPath runs EV fast chargers in five Indian cities: Jaipur, Delhi, Mumbai, Bengaluru, Pune.'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN station_id COMMENT 'Unique identifier for each charging station'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN station_name COMMENT 'Human-readable name of the station'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN city COMMENT 'City where the station is located (Jaipur, Delhi, Mumbai, Bengaluru, Pune)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN site_type COMMENT 'Type of site location (e.g. highway, mall, petrol pump)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN opened_on COMMENT 'Date the station was opened for public use'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN _source_file COMMENT 'Source file path from which the record was ingested'")
spark.sql(f"ALTER TABLE {catalog}.bronze.stations ALTER COLUMN _ingested_at COMMENT 'Timestamp when the record was loaded into bronze'")

# ── bronze.chargers ──
spark.sql(f"COMMENT ON TABLE {catalog}.bronze.chargers IS 'Raw charger data ingested from multi-line JSON weekly full snapshots. Each station has multiple chargers with connector type and max power output.'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN charger_id COMMENT 'Unique identifier for each charger'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN station_id COMMENT 'Foreign key to bronze.stations identifying which station this charger belongs to'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN connector COMMENT 'Struct containing connector type (CCS2 or TYPE2) and max_kw (maximum power output in kW)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN is_active COMMENT 'Whether the charger is currently operational (true/false)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN commissioned_on COMMENT 'Date the charger was commissioned and ready for use'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN snapshot_date COMMENT 'Date of the weekly full snapshot this record came from'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN _source_file COMMENT 'Source file path from which the record was ingested'")
spark.sql(f"ALTER TABLE {catalog}.bronze.chargers ALTER COLUMN _ingested_at COMMENT 'Timestamp when the record was loaded into bronze'")

# ── bronze.tariffs ──
spark.sql(f"COMMENT ON TABLE {catalog}.bronze.tariffs IS 'Raw tariff data ingested from a single CSV file. Defines pricing per connector type and time band (PEAK/OFF_PEAK) for billing calculations.'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN connector_type COMMENT 'Type of connector this tariff applies to (CCS2 or TYPE2)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN time_band COMMENT 'Time period for this tariff rate (PEAK or OFF_PEAK)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN rate_per_kwh COMMENT 'Energy charge rate per kWh consumed'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN idle_fee_per_min COMMENT 'Fee charged per minute of idle time after charging completes'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN free_idle_minutes COMMENT 'Number of free idle minutes before the idle fee kicks in'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN _source_file COMMENT 'Source file path from which the record was ingested'")
spark.sql(f"ALTER TABLE {catalog}.bronze.tariffs ALTER COLUMN _ingested_at COMMENT 'Timestamp when the record was loaded into bronze'")

# ── bronze.member ──
spark.sql(f"COMMENT ON TABLE {catalog}.bronze.member IS 'Raw member data ingested from CSV — one initial snapshot file followed by a change file. Members are on BASIC, PRO or FLEET plans which determine billing discounts. Guests have no membership.'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN member_id COMMENT 'Unique identifier for each registered member'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN full_name COMMENT 'Full name of the member'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN email COMMENT 'Email address of the member'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN phone COMMENT 'Phone number of the member'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN city COMMENT 'City where the member is located'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN plan COMMENT 'Membership plan (BASIC, PRO or FLEET) — determines discount on charging fees'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN joined_on COMMENT 'Date the member joined VoltPath'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN change_effective_date COMMENT 'Effective date for change records; null for the initial snapshot rows'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN _source_file COMMENT 'Source file path from which the record was ingested'")
spark.sql(f"ALTER TABLE {catalog}.bronze.member ALTER COLUMN _ingested_at COMMENT 'Timestamp when the record was loaded into bronze'")

# ── bronze.sessions ──
spark.sql(f"COMMENT ON TABLE {catalog}.bronze.sessions IS 'Raw charging session data ingested from JSON Lines files — one file per day (1-10 April 2026). A session can appear more than once due to late and updated records. Contains nested meter, payment and faults structures along with auth_method.'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN session_id COMMENT 'Unique identifier for each charging session'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN charger_id COMMENT 'Foreign key to bronze.chargers identifying which charger was used'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN member_id COMMENT 'Foreign key to bronze.member; null for guest (non-member) sessions'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN started_at COMMENT 'Start timestamp of the charging session'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN ended_at COMMENT 'End timestamp of the charging session'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN updated_at COMMENT 'Timestamp of the last update to the session record; used to identify the latest version of a session'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN status COMMENT 'Status of the session (COMPLETED, FAILED or STARTED)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN meter COMMENT 'Struct containing start_kwh and end_kwh meter readings used to calculate energy consumed'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN payment COMMENT 'Struct containing payment method (UPI, CARD, WALLET) and amount_inr (amount paid in Indian Rupees)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN faults COMMENT 'Array of structs containing fault code and timestamp for any faults during the session'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN idle_minutes COMMENT 'Minutes the vehicle remained idle after charging completed; billed per tariff idle fee'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN auth_method COMMENT 'How the session was authorised by the driver (app, qr, QR)'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN _rescued_data COMMENT 'Fields that did not match the Auto Loader schema and were rescued for data quality review'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN _source_file COMMENT 'Source file path from which the record was ingested'")
spark.sql(f"ALTER TABLE {catalog}.bronze.sessions ALTER COLUMN _ingested_at COMMENT 'Timestamp when the record was loaded into bronze'")

print("All bronze table and column comments added successfully.")